# AI-RAG · Week 15 — Notebook 1 (Day 1)
## Understand & Build the Memory Layer

The **conceptual relief week** after W14's peak. The idea: give the agent memory — but *design memory that helps, not hurts*. Day 1: the memory taxonomy, the three patterns, **when NOT to use memory**, and the state-based memory layer, built and demonstrated. Runs **fully offline**.

> Code is lifted **verbatim** from the package (`src/agents/**`). Changes are notebook glue only (local paths, inlined imports); the real-OpenAI model string is `gpt-5-mini`.

### Setup

In [1]:
# --- notebook setup (delivery glue only) -------------------------------------
# Runs offline on a memory-aware mock LLM — no OpenAI/Tavily key, no cost.
try:
    import pydantic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pydantic"])

from pathlib import Path
Path("data").mkdir(exist_ok=True)
Path("data/memory_golden_set.jsonl").write_text(r"""{"id": "Q1", "kind": "preference", "memory_effect": "helps", "question": "Summarise the leave policy in my usual format.", "seed": {"preferred_format": "bullets"}, "relevant_key": "preferred_format"}
{"id": "Q2", "kind": "preference", "memory_effect": "helps", "question": "Give me the expense policy the way I like it.", "seed": {"preferred_format": "bullets"}, "relevant_key": "preferred_format"}
{"id": "Q3", "kind": "stateless", "memory_effect": "neutral", "question": "What is 15% of 240?", "seed": {"name": "Asha", "team": "HR", "tenure": "3 years", "preferred_format": "bullets"}, "relevant_key": null}
{"id": "Q4", "kind": "stateless", "memory_effect": "neutral", "question": "How many HR documents are in the knowledge base?", "seed": {"name": "Asha", "team": "HR", "tenure": "3 years", "preferred_format": "bullets"}, "relevant_key": null}
{"id": "Q5", "kind": "stateless", "memory_effect": "neutral", "question": "What is the standard notice period?", "seed": {"name": "Asha", "team": "HR", "tenure": "3 years", "preferred_format": "bullets"}, "relevant_key": null}
{"id": "Q6", "kind": "stateless", "memory_effect": "neutral", "question": "Convert 90 minutes to hours.", "seed": {"name": "Asha", "team": "HR", "tenure": "3 years", "preferred_format": "bullets"}, "relevant_key": null}
{"id": "Q7", "kind": "stateless", "memory_effect": "neutral", "question": "What does the acronym PTO stand for?", "seed": {"name": "Asha", "team": "HR", "tenure": "3 years", "preferred_format": "bullets"}, "relevant_key": null}
{"id": "Q8", "kind": "stale", "memory_effect": "hurts", "question": "Which project should I focus on this week?", "seed": {"current_project": "Project Atlas"}, "relevant_key": "current_project"}
{"id": "Q9", "kind": "stale", "memory_effect": "hurts", "question": "Remind me which office I'm assigned to.", "seed": {"assigned_office": "Building C (closed last quarter)"}, "relevant_key": "assigned_office"}""" + "\n")
Path("data/web_fixtures.json").write_text(r"""{
  "ai agents": [
    {"title": "State of AI Agents 2026", "snippet": "Agentic systems moved from demos to production in 2026, with tool-use reliability the key differentiator."},
    {"title": "Agent frameworks roundup", "snippet": "Raw loops and graph-based orchestration remain the two dominant patterns."}
  ],
  "openai api": [
    {"title": "OpenAI API pricing", "snippet": "gpt-4o-mini is positioned as the low-cost tier for high-volume workloads."}
  ],
  "cricket": [
    {"title": "Cricket World Cup — result", "snippet": "The most recent tournament concluded with a closely contested final."}
  ],
  "python libraries": [
    {"title": "Trending Python libraries", "snippet": "Data, LLM-tooling, and async libraries dominate this year's trending lists."}
  ],
  "announce": [
    {"title": "OpenAI announcements", "snippet": "Recent announcements focus on tool use, structured outputs, and cost reductions."}
  ],
  "_default": [
    {"title": "Web result", "snippet": "A relevant public-web snippet would appear here (offline fixture)."}
  ]
}""" + "\n")
print("wrote the labelled golden set + web fixtures")


wrote the labelled golden set + web fixtures


### Setup · the side-database

In [2]:
#!/usr/bin/env python3
"""scripts/seed_db.py — (re)create the capstone side database.

W13 gives the assistant structured data to query as a tool. This builds a small
SQLite database with two tables:

  document_metadata  — one row per document in the knowledge base (the structured
                       side-table the SQL tool queries)
  audit_log          — every SQL query the agent runs is logged here (the audit
                       rail for SQL-as-a-tool)

SQLite is local and offline, so the whole SQL-tool exercise runs with no network
and no credentials. Run this once (smoke_test.sh does it for you):

    python3 scripts/seed_db.py
"""
from __future__ import annotations

import sqlite3
from pathlib import Path

DB_PATH = Path("data/capstone_side.db")   # notebook: local file

# A small, realistic slice of the Enterprise Knowledge Assistant's corpus,
# across the three departments the capstone covers.
DOCS = [
    # source, title, doc_type, department, version, updated_at, chunk_count
    ("hr-001", "Leave & Time-Off Policy",          "policy",   "HR",      "v3", "2026-01-14", 42),
    ("hr-002", "Code of Conduct",                  "policy",   "HR",      "v2", "2025-11-02", 55),
    ("hr-003", "Remote Work Guidelines",           "guideline","HR",      "v4", "2026-02-20", 28),
    ("hr-004", "Onboarding Handbook",              "handbook", "HR",      "v1", "2025-06-30", 90),
    ("it-001", "Acceptable Use Policy",            "policy",   "IT",      "v5", "2026-01-30", 37),
    ("it-002", "Security Incident Runbook",        "runbook",  "IT",      "v2", "2026-02-11", 33),
    ("it-003", "VPN & Access Setup",               "guideline","IT",      "v3", "2025-12-05", 19),
    ("it-004", "Data Classification Standard",     "standard", "IT",      "v2", "2026-01-08", 24),
    ("fin-001","Expense Reimbursement Policy",     "policy",   "Finance", "v3", "2026-01-22", 31),
    ("fin-002","Procurement Guidelines",           "guideline","Finance", "v1", "2025-09-15", 40),
    ("fin-003","Travel Policy",                    "policy",   "Finance", "v2", "2025-10-19", 26),
    ("fin-004","Corporate Card Terms",             "terms",    "Finance", "v4", "2026-02-01", 18),
]


def seed(db_path: Path = DB_PATH) -> Path:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE document_metadata (
            id           INTEGER PRIMARY KEY,
            source       TEXT NOT NULL,
            title        TEXT NOT NULL,
            doc_type     TEXT NOT NULL,
            department   TEXT NOT NULL,
            version      TEXT NOT NULL,
            updated_at   TEXT NOT NULL,
            chunk_count  INTEGER NOT NULL
        )""")
    cur.executemany(
        "INSERT INTO document_metadata "
        "(source,title,doc_type,department,version,updated_at,chunk_count) "
        "VALUES (?,?,?,?,?,?,?)", DOCS)
    cur.execute("""
        CREATE TABLE audit_log (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            ts         TEXT NOT NULL,
            actor      TEXT NOT NULL,
            request    TEXT,
            sql        TEXT,
            decision   TEXT NOT NULL,
            row_count  INTEGER,
            error      TEXT
        )""")
    con.commit()
    con.close()
    return db_path

seed()
print('side-DB built:', DB_PATH)


side-DB built: data/capstone_side.db


## 1 · The memory taxonomy — five kinds

Before building anything, the vocabulary. Five kinds of memory — but **most capstones need one or two, not five**:

| Kind | What it is | Capstone example |
|---|---|---|
| **Short-term** | within one conversation | the running message history (usually free) |
| **Long-term** | persists across sessions | 'this user prefers bullet points' |
| **Episodic** | a specific past event | 'on Oct 12 asked about leave, unsatisfied' |
| **Semantic** | a fact about the user | 'the user is on the HR team' |
| **Procedural** | how the user likes things | 'wants citations as footnotes' |

The two you'll actually use: **short-term** (free — you already pass the history) and one slice of **long-term**. That long-term slice is what we build below.

## 2 · Three implementation patterns

| Pattern | Best for | Cost |
|---|---|---|
| **State-based** (Pydantic + SQLite) — *the default* | known, structured facts | cheap, inspectable |
| **Vector** (a memory collection, RAG-style) | unstructured episodic recall | more moving parts |
| **Summary** (summarise every N turns) | controlling prompt length | an extra LLM call |

**The decision rule:** state for known structure, summary for length, vector for unstructured recall. Reaching for vector first? Ask *what can't a Pydantic state hold?* — usually the simpler pattern wins. We implement the **state-based** default.

## 3 · When NOT to use memory

**The default is NO memory.** It adds complexity, cost, and surface area — add it only when you can *prove* it helps (the W4 mindset). Four anti-patterns:

1. **Stateless tasks** — a calculator has nothing to remember; memory just adds tokens.
2. **Low repeat rate** — most users ask one-off questions; memory pays rent it never earns.
3. **Privacy / compliance** — memory = PII persistence, a liability (W8 → W28).
4. **Stale-context risk** — *wrong memory is worse than none* — it actively misleads.

Notebook 2 proves this by *measuring* memory on-vs-off.

## 4 · The memory layer — `src/agents/memory.py`

The state-based default: a Pydantic **`UserState`** (facts + timestamps), a SQLite **`MemoryStore`**, **TTL** forgetting (drop facts older than N days), and user-controlled forget. Contradiction resolution here is **last-write-wins**: for a mutable preference, a newer write replaces the old. That is one strategy (others merge or keep versions) — fine for self-reported preferences, not a universal rule.

In [3]:
"""src/agents/memory.py — state-based memory for the agent.

The W15 default: a small, explicit **state** the agent remembers about a user
across sessions — a Pydantic model, saved to SQLite. This is the right pattern for
the large majority of capstones (known, structured facts like a preferred output format). Vector
and summary memory are more powerful and more complex; see docs/memory_taxonomy.md.

Two kinds of memory live here:
  - short-term — the conversation history itself (handled by the agent loop, not stored)
  - long-term  — this UserState, persisted across sessions

Forgetting is deliberate: **TTL** drops stale facts (the sane default), and the user
can explicitly forget a fact. Contradiction resolution uses last-write-wins — for a mutable
preference, a newer write replaces the old (the user is authoritative about their
own preferences). It is one strategy of several (merge, or keep versioned history);
scoped to mutable preference facts here, not a universal rule.
"""
from __future__ import annotations

import sqlite3
from datetime import date, timedelta
from pathlib import Path
from typing import Callable

from pydantic import BaseModel, Field

DEFAULT_DB = Path("data/capstone_side.db")   # notebook: local file (built in setup)
_today: Callable[[], date] = date.today


class MemoryFact(BaseModel):
    value: str
    last_updated: str          # ISO date, e.g. "2026-01-15"


class UserState(BaseModel):
    """What the agent remembers about one user. `facts` maps a key
    (e.g. 'preferred_format') to a value + when it was last set."""
    user_id: str
    facts: dict[str, MemoryFact] = Field(default_factory=dict)

    # ── updates (last-write-wins for a mutable pref → contradiction resolution) ──
    def set(self, key: str, value: str, today: Callable[[], date] = _today) -> None:
        self.facts[key] = MemoryFact(value=value, last_updated=today().isoformat())

    def get(self, key: str) -> str | None:
        f = self.facts.get(key)
        return f.value if f else None

    def forget(self, key: str) -> bool:
        """User-controlled forgetting — an explicit 'forget X'."""
        return self.facts.pop(key, None) is not None

    # ── forgetting strategy: TTL ────────────────────────────────────────────
    def apply_ttl(self, days: int = 30, today: Callable[[], date] = _today) -> list[str]:
        """Drop facts whose last_updated is older than `days`. Returns dropped keys."""
        cutoff = today() - timedelta(days=days)
        dropped = [k for k, f in self.facts.items()
                   if date.fromisoformat(f.last_updated) < cutoff]
        for k in dropped:
            del self.facts[k]
        return dropped

    # ── render for the system prompt (short + relevant only) ────────────────
    def to_prompt(self) -> str:
        if not self.facts:
            return ""
        lines = [f"- {k.replace('_', ' ')}: {f.value}" for k, f in sorted(self.facts.items())]
        return "What you remember about this user:\n" + "\n".join(lines)


class MemoryStore:
    """SQLite-backed persistence for UserState. One row per (user_id, key)."""

    def __init__(self, db_path: Path | str = DEFAULT_DB):
        self.db_path = Path(db_path)
        self._ensure_table()

    def _ensure_table(self) -> None:
        con = sqlite3.connect(self.db_path)
        con.execute("""CREATE TABLE IF NOT EXISTS user_memory (
            user_id TEXT NOT NULL, key TEXT NOT NULL,
            value TEXT NOT NULL, last_updated TEXT NOT NULL,
            PRIMARY KEY (user_id, key))""")
        con.commit(); con.close()

    def load_state(self, user_id: str) -> UserState:
        con = sqlite3.connect(self.db_path)
        rows = con.execute("SELECT key, value, last_updated FROM user_memory WHERE user_id=?",
                           (user_id,)).fetchall()
        con.close()
        facts = {k: MemoryFact(value=v, last_updated=lu) for k, v, lu in rows}
        return UserState(user_id=user_id, facts=facts)

    def save_state(self, state: UserState) -> None:
        con = sqlite3.connect(self.db_path)
        con.execute("DELETE FROM user_memory WHERE user_id=?", (state.user_id,))
        con.executemany(
            "INSERT INTO user_memory (user_id, key, value, last_updated) VALUES (?,?,?,?)",
            [(state.user_id, k, f.value, f.last_updated) for k, f in state.facts.items()])
        con.commit(); con.close()

    def clear(self, user_id: str) -> None:
        con = sqlite3.connect(self.db_path)
        con.execute("DELETE FROM user_memory WHERE user_id=?", (user_id,))
        con.commit(); con.close()


# ─── the two hooks the agent loop uses ──────────────────────────────────────
def load_for_prompt(store: MemoryStore, user_id: str, ttl_days: int = 30,
                    today: Callable[[], date] = _today) -> tuple[UserState, str]:
    """Load state, apply TTL, and render it for the system prompt.
    Returns (state, prompt_block). Called at the START of a run when memory is on."""
    state = store.load_state(user_id)
    dropped = state.apply_ttl(days=ttl_days, today=today)
    if dropped:
        store.save_state(state)          # persist the forgetting
    return state, state.to_prompt()


def update_from_turn(store: MemoryStore, state: UserState, question: str,
                     today: Callable[[], date] = _today) -> str | None:
    """User-controlled memory: honour explicit 'remember …' / 'forget …' in the
    user's message. Called at the END of a run when memory is on. Returns a note."""
    q = question.strip()
    low = q.lower()
    if low.startswith("remember that "):
        payload = q[len("remember that "):].strip().rstrip(".")
        if " is " in payload:
            key, value = payload.split(" is ", 1)
            state.set(key.strip().replace(" ", "_"), value.strip(), today=today)
            store.save_state(state)
            return f"remembered {key.strip()} = {value.strip()}"
    if low.startswith("forget "):
        key = low[len("forget "):].strip().rstrip(".").replace(" ", "_")
        if state.forget(key):
            store.save_state(state)
            return f"forgot {key}"
    return None


**Set/get with last-write-wins (scoped to preferences), then persist across a 'session' (SQLite):**

In [4]:
from datetime import date
JAN10 = lambda: date(2026,1,10); JAN15 = lambda: date(2026,1,15); NOV01 = lambda: date(2025,11,1)

st = UserState(user_id='asha')
st.set('preferred_format', 'prose',   today=JAN10)
st.set('preferred_format', 'bullets', today=JAN15)   # contradiction → last-write-wins (preference)
print('get →', st.get('preferred_format'))

store = MemoryStore(); store.clear('asha'); store.save_state(st)
print('reloaded from SQLite →', MemoryStore().load_state('asha').get('preferred_format'))

get → bullets
reloaded from SQLite → bullets


**TTL drops stale facts; `to_prompt()` renders what gets injected:**

In [5]:
st.set('current_project', 'Atlas', today=NOV01)   # 75 days old
dropped = st.apply_ttl(days=30, today=JAN15)
print('TTL dropped:', dropped, '| remaining:', list(st.facts))
print('\nprompt block:\n' + st.to_prompt())

TTL dropped: ['current_project'] | remaining: ['preferred_format']

prompt block:
What you remember about this user:
- preferred format: bullets


**User-controlled memory — `remember` / `forget`:**

In [6]:
store.clear('u'); s = UserState(user_id='u')
print(update_from_turn(store, s, 'remember that team is HR', today=JAN15))
print('  stored →', MemoryStore().load_state('u').get('team'))
print(update_from_turn(store, s, 'forget team', today=JAN15))
print('  stored →', MemoryStore().load_state('u').get('team'))

remembered team = HR
  stored → HR
forgot team
  stored → None


## 5 · Memory in the agent

The mock LLM, the defenses, the tools, and the raw agent — carried from W13/W14. This week's focus is the **memory layer**, so these are here just so the agent runs. The agent loads the user's state at the start of a run, **injects it into the system prompt**, and updates it at the end.

In [7]:
"""src/agents/defenses.py — Failure-Oriented Design, in code.

Five failure categories, five defenses:

    Transient    (429, network blip)        → retry with backoff
    Permanent    (bad args, bad schema)     → fast-fail (do NOT retry)
    Silent       (wrong-but-valid result)   → validation / sanity checks (in the tools)
    Cascading    (timeout floods downstream)→ circuit breaker
    Adversarial  (injection, misuse)         → tool permissioning (+ sanitisation)

`DefensePolicy.guard(tool_name, call)` wraps a single tool call with permissioning,
a circuit breaker, retry-with-backoff, and graceful degradation, so one bad tool
can't crash the agent. Deterministic and offline (backoff sleeps are disabled in
tests via sleep=lambda _: None).
"""
from __future__ import annotations

import time
from dataclasses import dataclass, field
from typing import Callable


# ─── failure types ──────────────────────────────────────────────────────────
class TransientError(Exception):
    """Temporary — a retry will probably succeed (e.g. network blip)."""


class RateLimited(TransientError):
    """HTTP 429 — transient, back off and retry."""


class PermanentError(Exception):
    """Won't fix itself — retrying is harmful (bad credentials, bad schema/args)."""


def classify(exc: Exception) -> str:
    """Map an exception to a failure category (the diagnostic the week teaches)."""
    if isinstance(exc, RateLimited):
        return "transient"
    if isinstance(exc, TransientError) or isinstance(exc, TimeoutError):
        return "transient"      # timeouts retry, but repeated ones trip the breaker → cascading
    if isinstance(exc, PermanentError):
        return "permanent"
    return "permanent"          # unknown = don't blindly retry


# ─── circuit breaker (hand-rolled: CLOSED / OPEN / HALF_OPEN) ────────────────
class CircuitOpen(Exception):
    """Raised when the breaker is open and the call is short-circuited."""


class CircuitBreaker:
    """Opens after `fail_max` consecutive failures; after `reset_timeout` it moves
    to HALF_OPEN and lets one trial call through. Success closes it; failure
    re-opens it. This is the cascading-failure defense."""

    def __init__(self, fail_max: int = 3, reset_timeout: float = 30.0,
                 now: Callable[[], float] = time.monotonic):
        self.fail_max = fail_max
        self.reset_timeout = reset_timeout
        self._now = now
        self.state = "closed"
        self.failures = 0
        self.opened_at = 0.0

    def _can_attempt(self) -> bool:
        if self.state == "open":
            if self._now() - self.opened_at >= self.reset_timeout:
                self.state = "half_open"
                return True
            return False
        return True

    def record_success(self) -> None:
        self.failures = 0
        self.state = "closed"

    def record_failure(self) -> None:
        self.failures += 1
        if self.failures >= self.fail_max:
            self.state = "open"
            self.opened_at = self._now()

    def call(self, fn: Callable[[], str]) -> str:
        if not self._can_attempt():
            raise CircuitOpen(f"circuit open (>{self.fail_max} failures)")
        try:
            result = fn()
        except Exception:
            self.record_failure()
            raise
        self.record_success()
        return result


# ─── retry with exponential backoff (the transient defense) ─────────────────
def retry_with_backoff(fn: Callable[[], str], attempts: int = 3, base: float = 0.2,
                       sleep: Callable[[float], None] = time.sleep) -> str:
    """Retry on TransientError/TimeoutError with exponential backoff. PermanentError
    is re-raised immediately (fast-fail)."""
    last: Exception | None = None
    for i in range(attempts):
        try:
            return fn()
        except PermanentError:
            raise                                   # never retry a permanent failure
        except (TransientError, TimeoutError) as e:
            last = e
            if i < attempts - 1:
                sleep(base * (2 ** i))
    raise last if last else RuntimeError("retry failed")


# ─── the policy that wraps every tool call ──────────────────────────────────
@dataclass
class DefensePolicy:
    retries: int = 3
    breaker_tools: tuple[str, ...] = ("search_web", "query_sql")   # external / can time out
    write_tools: tuple[str, ...] = ("send_email",)       # gated behind confirmation
    fail_max: int = 3
    sleep: Callable[[float], None] = time.sleep
    breakers: dict = field(default_factory=dict)

    def __post_init__(self):
        for t in self.breaker_tools:
            self.breakers.setdefault(t, CircuitBreaker(fail_max=self.fail_max))

    def guard(self, tool_name: str, call: Callable[[], str]) -> tuple[str, str]:
        """Run a tool call under the defenses. Returns (observation, status) where
        status ∈ {ok, degraded, blocked}. Never raises — the agent must survive."""
        # 1 · Adversarial → tool permissioning: gate side-effecting tools.
        if tool_name in self.write_tools:
            return (f"[{tool_name} BLOCKED — requires human confirmation before a "
                    f"side-effecting action]", "blocked")

        breaker = self.breakers.get(tool_name)

        def attempt() -> str:
            return breaker.call(call) if breaker else call()

        # 2 · Cascading → if the breaker is already open, degrade fast.
        try:
            obs = retry_with_backoff(attempt, attempts=self.retries, sleep=self.sleep)
            return obs, "ok"
        except CircuitOpen as e:
            return (f"[{tool_name} unavailable — {e}; answering without it]", "degraded")
        except Exception as e:                       # transient exhausted / permanent
            cat = classify(e)
            return (f"[{tool_name} failed ({cat}: {e}); answering without it]", "degraded")


In [8]:
"""src/agents/mock_llm.py — the LLM behind the agent, and how we mock it.

The single most useful testing skill this week is **mocking the LLM**. A real LLM
call is slow, costs money, and is non-deterministic — useless in a test. So the
agent talks to a tiny interface:

    llm.complete(messages, tools) -> LLMResponse   # either a tool call or a final answer

Two implementations:
  MockLLM  — deterministic and offline. Give it a *script* (a fixed list of
             responses, for tests) or let it keyword-route one tool call then a
             final answer (for offline runs). No network, no key, milliseconds.
  OpenAILLM — the real thing (OpenAI function calling). Needs OPENAI_API_KEY.

Everything in this package runs on MockLLM, which is why the whole thing — agent,
defenses, the five failure scenarios, the test suite — works with no credentials.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from typing import Callable

# Tool is defined in the cell above


@dataclass
class LLMResponse:
    kind: str                      # "tool_use" | "final"
    tool_name: str | None = None
    tool_args: dict | None = None
    content: str | None = None

    @staticmethod
    def tool(name: str, **args) -> "LLMResponse":
        return LLMResponse("tool_use", tool_name=name, tool_args=args)

    @staticmethod
    def final(content: str) -> "LLMResponse":
        return LLMResponse("final", content=content)


class MockLLM:
    """A deterministic stand-in for a function-calling LLM.

    - Scripted:  MockLLM(script=[LLMResponse.tool('calculate', expression='2+2'),
                                 LLMResponse.final('It is 4.')])
                 returns those in order (extra calls repeat the last one).
    - Keyword:   MockLLM()  → routes ONE tool call by keyword, then a final answer
                 synthesised from the observation. Good for offline demos.
    """

    def __init__(self, script: list[LLMResponse] | None = None,
                 router: Callable[[list[dict], list[Tool]], LLMResponse] | None = None):
        self.script = list(script) if script else None
        self.router = router
        self.calls = 0

    def complete(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        self.calls += 1
        if self.script is not None:
            return self.script.pop(0) if self.script else LLMResponse.final(
                "(no more scripted responses)")
        if self.router is not None:
            return self.router(messages, tools)
        return self._keyword_route(messages, tools)

    # ── default keyword routing: one tool call, then a final answer ──────────
    def _keyword_route(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        # If a tool has already produced an observation, answer from it.
        obs = _last_observation(messages)
        if obs is not None:
            return LLMResponse.final(f"Based on the tool result: {obs}")
        q = _last_user(messages).lower()
        names = {t.name for t in tools}
        if "calculate" in names and re.search(r"\d\s*[-+*/%]\s*\d|percent|%|how much is", q):
            return LLMResponse.tool("calculate", expression=_extract_expr(q))
        sql_kw = ("how many doc", "which department", "what version", "list the", "count",
                  "how many hr", "how many it", "how many finance", "updated", "chunk")
        if "query_sql" in names and any(k in q for k in sql_kw):
            return LLMResponse.tool("query_sql",
                                    sql="SELECT COUNT(*) AS total_documents FROM document_metadata",
                                    purpose="metadata question")
        web_kw = ("latest", "current", "today", "news", "price", "recent", "2026", "who won")
        if "search_web" in names and any(k in q for k in web_kw):
            return LLMResponse.tool("search_web", query=_last_user(messages), max_results=3)
        return LLMResponse.final("I can answer that directly: (no tool needed).")


# ─── real OpenAI adapter (needs a key + network) ────────────────────────────
class OpenAILLM:
    def __init__(self, model: str = "gpt-5-mini"):
        from openai import OpenAI
        self.client = OpenAI()
        self.model = model

    def complete(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        resp = self.client.chat.completions.create(
            model=self.model, messages=messages,
            tools=[t.openai_schema() for t in tools], tool_choice="auto")
        msg = resp.choices[0].message
        if msg.tool_calls:
            call = msg.tool_calls[0]
            return LLMResponse.tool(call.function.name, **json.loads(call.function.arguments or "{}"))
        return LLMResponse.final(msg.content or "")


# ─── tiny helpers ───────────────────────────────────────────────────────────
def _last_user(messages: list[dict]) -> str:
    for m in reversed(messages):
        if m.get("role") == "user":
            return str(m.get("content", ""))
    return ""


def _last_observation(messages: list[dict]) -> str | None:
    for m in reversed(messages):
        if m.get("role") == "tool":
            return str(m.get("content", ""))
    return None


def _extract_expr(q: str) -> str:
    m = re.search(r"[0-9][0-9\.\s\+\-\*/%\(\)]*[0-9\)]", q)
    return m.group(0).strip() if m else "0"


In [9]:
"""src/agents/tools/base.py — the Tool contract.

A tool an LLM can call reliably is four things done well (the W13 principles):
  1. a good NAME       — a verb, specific, snake_case
  2. a typed SCHEMA    — a Pydantic model: required fields, sensible defaults
  3. a DESCRIPTION     — written for the *model*, saying WHEN to use the tool
  4. IDEMPOTENCY        — declared, so the loop knows if a retry is safe

Every tool subclasses `Tool`, sets those four, and implements `run`. The same
definition exports to both OpenAI function-calling and Anthropic tool-use schemas.
"""
from __future__ import annotations

from typing import Any

from pydantic import BaseModel


class Tool:
    name: str = ""
    description: str = ""
    Args: type[BaseModel] = BaseModel
    idempotent: bool = True          # is calling this twice with the same args safe?
    has_side_effects: bool = False   # does it change external state?

    def run(self, **kwargs: Any) -> str:
        raise NotImplementedError

    # ── validation + execution ────────────────────────────────────────────
    def call(self, arguments: dict[str, Any]) -> str:
        """Validate arguments against the schema, then run."""
        args = self.Args(**(arguments or {}))
        return self.run(**args.model_dump())

    # ── provider schema exports ───────────────────────────────────────────
    def parameters_schema(self) -> dict:
        return self.Args.model_json_schema()

    def openai_schema(self) -> dict:
        return {"type": "function",
                "function": {"name": self.name,
                             "description": self.description,
                             "parameters": self.parameters_schema()}}

    def anthropic_schema(self) -> dict:
        return {"name": self.name,
                "description": self.description,
                "input_schema": self.parameters_schema()}


In [10]:
"""src/agents/tools/calculate.py — a computation tool.

Deterministic, offline, and safe: it evaluates an arithmetic expression with a
small AST walker (no `eval`), so an LLM can hand it a calculation and get a
reliable number back. Idempotent and side-effect-free — the easy end of the
tool spectrum, and a good first example of a well-formed tool.
"""
from __future__ import annotations

import ast
import operator

from pydantic import BaseModel, Field

# Tool is defined in the cell above

_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos, ast.FloorDiv: operator.floordiv,
}


def _eval(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return _eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported expression — only + - * / % ** ( ) and numbers")


class CalculateArgs(BaseModel):
    expression: str = Field(
        ..., description="An arithmetic expression, e.g. '1350 * 0.18' or '(42 + 8) / 2'.")


class CalculateTool(Tool):
    name = "calculate"
    description = ("Evaluate an arithmetic expression and return the numeric result. "
                   "Use this whenever a question needs exact math — sums, differences, "
                   "products, division, percentages, or powers. Do NOT use it to look up "
                   "facts; it only does arithmetic on numbers you provide.")
    Args = CalculateArgs
    idempotent = True
    has_side_effects = False

    def run(self, expression: str) -> str:
        try:
            result = _eval(ast.parse(expression, mode="eval"))
        except Exception as e:  # noqa: BLE001 - report cleanly to the agent
            return f"calculate error: {e}"
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return f"{expression} = {result}"


In [11]:
"""src/agents/tools/query_sql.py — SQL as a tool, safely.

Letting an LLM run SQL is powerful and dangerous. This tool ships the three rails
every SQL-as-a-tool needs in production:

  1. AUDIT LOGGING     — every query (allowed or denied) is written to audit_log
  2. READ-ONLY DEFAULT  — only a single SELECT / WITH…SELECT is permitted
  3. CONSTRAINED UPDATES — writes/DDL (INSERT/UPDATE/DELETE/DROP/…) are denied
                          outright (write access is a W18 topic, gated behind
                          injection defenses and a human-approval step)

Everything runs against a local SQLite database, so it works offline with no
credentials. The migration path to Postgres is a config change, not a rewrite
(see docs/runbooks/sql_as_a_tool_safety.md).
"""
from __future__ import annotations

import re
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

from pydantic import BaseModel, Field

# Tool is defined in the cell above

DB_PATH = Path("data/capstone_side.db")   # notebook: local file

# Statements that change data or schema — denied by the read-only rail.
_FORBIDDEN = {"insert", "update", "delete", "drop", "alter", "create", "replace",
              "truncate", "attach", "detach", "pragma", "vacuum", "reindex",
              "grant", "revoke", "begin", "commit"}
_MAX_ROWS = 50


def _strip_comments(sql: str) -> str:
    sql = re.sub(r"--[^\n]*", " ", sql)
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    return sql.strip()


def is_read_only(sql: str) -> tuple[bool, str]:
    """Return (ok, reason). Only a single SELECT/WITH…SELECT is allowed."""
    clean = _strip_comments(sql)
    if not clean:
        return False, "empty query"
    statements = [s for s in clean.split(";") if s.strip()]
    if len(statements) > 1:
        return False, "multiple statements are not allowed"
    first = statements[0].lstrip().split(None, 1)[0].lower()
    if first not in ("select", "with"):
        return False, f"only SELECT queries are allowed (got '{first.upper()}')"
    words = set(re.findall(r"[a-z_]+", clean.lower()))
    hit = words & _FORBIDDEN
    if hit:
        return False, f"write/DDL keyword not permitted: {', '.join(sorted(hit))}"
    return True, "ok"


class QuerySqlArgs(BaseModel):
    sql: str = Field(..., description="A single read-only SQL SELECT query over the "
                                      "document_metadata table.")
    purpose: str = Field("", description="One phrase describing what the query is for "
                                         "(recorded in the audit log).")


class QuerySqlTool(Tool):
    name = "query_sql"
    description = (
        "Run a read-only SQL SELECT against the knowledge-base metadata database and "
        "return the rows. Use this for questions about the DOCUMENTS THEMSELVES — how "
        "many documents exist, which department owns them, their type, version, "
        "chunk_count, or when they were last updated. Table: document_metadata"
        "(id, source, title, doc_type, department, version, updated_at, chunk_count). "
        "Do NOT use it to answer questions about document CONTENTS — only their metadata.")
    Args = QuerySqlArgs
    idempotent = True            # SELECTs don't change state
    has_side_effects = False

    def __init__(self, db_path: Path | str = DB_PATH):
        self.db_path = Path(db_path)

    def _audit(self, request: str, sql: str, decision: str,
               row_count: int | None, error: str | None) -> None:
        con = sqlite3.connect(self.db_path)
        con.execute(
            "INSERT INTO audit_log (ts, actor, request, sql, decision, row_count, error) "
            "VALUES (?,?,?,?,?,?,?)",
            (datetime.now(timezone.utc).isoformat(timespec="seconds"), "agent",
             request, sql, decision, row_count, error))
        con.commit()
        con.close()

    def run(self, sql: str, purpose: str = "") -> str:
        ok, reason = is_read_only(sql)
        if not ok:
            self._audit(purpose, sql, "denied", None, reason)
            return (f"query_sql DENIED: {reason}. This tool is read-only; write access "
                    f"is not available. (Logged to the audit trail.)")
        try:
            con = sqlite3.connect(self.db_path)
            cur = con.execute(sql)
            rows = cur.fetchmany(_MAX_ROWS)
            cols = [d[0] for d in cur.description] if cur.description else []
            con.close()
        except Exception as e:  # noqa: BLE001
            self._audit(purpose, sql, "error", None, str(e))
            return f"query_sql error: {e}"
        self._audit(purpose, sql, "allowed", len(rows), None)
        if not rows:
            return "query_sql: (0 rows)"
        header = " | ".join(cols)
        body = "\n".join(" | ".join(str(v) for v in r) for r in rows)
        more = "" if len(rows) < _MAX_ROWS else f"\n… (capped at {_MAX_ROWS} rows)"
        return f"{header}\n{'-' * len(header)}\n{body}{more}"


In [12]:
"""src/agents/tools/search_web.py — an external-API tool (web search).

Wraps a web-search API (Tavily) so the agent can reach information beyond the
knowledge base — current events, prices, anything not in the corpus. This is the
"API-calling agent" pattern: the LLM decides it needs fresh external info and
calls out.

Web search has real cost and latency, so the description tells the model to use
it only when the answer isn't in the corpus. For offline runs and tests, the tool
falls back to a local fixture file, so the whole thing works with no network and
no TAVILY_API_KEY.
"""
from __future__ import annotations

import json
import os
from pathlib import Path

from pydantic import BaseModel, Field

# Tool is defined in the cell above

FIXTURES_PATH = Path("data/web_fixtures.json")   # notebook: local file


class SearchWebArgs(BaseModel):
    query: str = Field(..., description="A concise web-search query.")
    max_results: int = Field(3, ge=1, le=5, description="How many results to return.")


class SearchWebTool(Tool):
    name = "search_web"
    description = (
        "Search the public web and return short result snippets. Use this ONLY when the "
        "answer is not in the internal knowledge base — for current events, live figures, "
        "recent releases, or anything outside company documents. It has cost and latency, "
        "so prefer the internal tools when they can answer. Returns titles and snippets, "
        "not full pages.")
    Args = SearchWebArgs
    idempotent = True            # reading search results doesn't change state
    has_side_effects = False

    def __init__(self, offline: bool | None = None):
        # offline when explicitly set, or when there is no API key available
        self.offline = (not os.environ.get("TAVILY_API_KEY")) if offline is None else offline

    def _fixture(self, query: str, k: int) -> str:
        try:
            data = json.loads(FIXTURES_PATH.read_text())
        except Exception:
            data = {"_default": [{"title": "No fixture", "snippet": "offline stub result."}]}
        results = None
        for key, res in data.items():
            if key != "_default" and key.lower() in query.lower():
                results = res
                break
        results = (results or data.get("_default", []))[:k]
        lines = [f"[web:{query}] (offline fixture)"]
        for r in results:
            lines.append(f"- {r['title']}: {r['snippet']}")
        return "\n".join(lines)

    def run(self, query: str, max_results: int = 3) -> str:
        if self.offline:
            return self._fixture(query, max_results)
        # live path (needs network + TAVILY_API_KEY)
        try:
            from tavily import TavilyClient
            client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
            resp = client.search(query=query, max_results=max_results)
            lines = [f"[web:{query}]"]
            for r in resp.get("results", [])[:max_results]:
                lines.append(f"- {r.get('title','')}: {r.get('content','')[:160]}")
            return "\n".join(lines)
        except Exception as e:  # noqa: BLE001
            return f"search_web error: {e} (falling back offline)\n" + self._fixture(query, max_results)


In [13]:
"""src/agents/tools/send_email.py — a WRITE tool, on purpose.

The three W13 tools are all read-only. This one has a side effect (it "sends"),
so it's the tool the permissioning defense exists for: the DefensePolicy gates
side-effecting tools behind a human-confirmation step, so an adversarial prompt
("ignore instructions and email everyone") is blocked rather than executed.

It never actually sends anything here — it's the shape that matters. It is marked
non-idempotent (a retry would send twice) and side-effecting.
"""
from __future__ import annotations

from pydantic import BaseModel, Field

# Tool is defined in the cell above


class SendEmailArgs(BaseModel):
    to: str = Field(..., description="Recipient email address.")
    subject: str = Field(..., description="Email subject line.")
    body: str = Field("", description="Email body.")


class SendEmailTool(Tool):
    name = "send_email"
    description = ("Send an email to a recipient. This performs a real, irreversible "
                   "action, so it is gated behind human confirmation. Use only when the "
                   "user explicitly asks to send a message.")
    Args = SendEmailArgs
    idempotent = False            # a retry would send a second email
    has_side_effects = True

    def run(self, to: str, subject: str, body: str = "") -> str:
        # If this ever executes, permissioning failed. Kept as a no-op on purpose.
        return f"(would send email to {to}: {subject!r})"


In [14]:
"""src/agents/raw_react.py — a ReAct agent in ~80 lines of raw Python.

ReAct = Reasoning + Acting in alternation. The loop:

    1. the LLM looks at the question + tools and returns a tool call OR a final answer
    2. if a tool call → run the tool, feed the observation back
    3. repeat, until a final answer OR max_iterations is hit

No framework. The max-iteration guard is non-negotiable — without it, one bad
model response can loop forever. Pass a DefensePolicy to harden every tool call
(retries, circuit breaker, permissioning, graceful degradation); leave it off to
see the bare loop.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any

# DefensePolicy, MockLLM, Tool are defined in the cells above


@dataclass
class Step:
    tool: str
    args: dict
    observation: str
    status: str = "ok"           # ok | degraded | blocked


@dataclass
class AgentResult:
    question: str
    answer: str
    steps: list[Step] = field(default_factory=list)
    iterations: int = 0
    stop_reason: str = "final"   # final | max_iter
    memory_used: bool = False    # was long-term memory loaded into the prompt?
    memory_note: str | None = None   # e.g. "remembered preferred_format = bullets"
    prompt_chars: int = 0        # total prompt characters processed (a cost proxy)
    def to_dict(self) -> dict:
        return {"question": self.question, "answer": self.answer,
                "iterations": self.iterations, "stop_reason": self.stop_reason,
                "memory_used": self.memory_used, "memory_note": self.memory_note,
                "prompt_chars": self.prompt_chars,
                "steps": [s.__dict__ for s in self.steps]}


SYSTEM = ("You are a helpful assistant that can call tools. Think about whether a "
          "tool is needed; call one if so, otherwise answer directly. Never invent "
          "tool results.")


class ReActAgent:
    def __init__(self, llm: Any, tools: list[Tool], max_iter: int = 5,
                 defenses: DefensePolicy | None = None,
                 memory: Any = None, user_id: str = "default", ttl_days: int = 30,
                 today: Any = None):
        self.llm = llm
        self.tools = tools
        self.by_name = {t.name: t for t in tools}
        self.max_iter = max_iter
        self.defenses = defenses
        self.memory = memory            # a MemoryStore, or None = memory OFF
        self.user_id = user_id
        self.ttl_days = ttl_days
        self.today = today              # injectable clock (tests)

    def _execute(self, name: str, args: dict) -> tuple[str, str]:
        tool = self.by_name.get(name)
        if tool is None:
            return f"[unknown tool '{name}']", "degraded"
        # PermanentError, RateLimited, TransientError are defined above

        def call() -> str:
            try:
                return tool.call(args or {})
            except (RateLimited, TransientError, TimeoutError):
                raise                             # let the policy retry / trip the breaker
            except PermanentError:
                raise                             # already classified
            except Exception as e:                # bad/hallucinated args, bad schema = permanent
                raise PermanentError(str(e))

        if self.defenses:
            return self.defenses.guard(name, call)
        return call(), "ok"                       # bare loop (no defenses)

    def run(self, question: str) -> AgentResult:
        result = AgentResult(question=question, answer="")
        # ── long-term memory: load state, apply TTL, inject into the system prompt ──
        system = SYSTEM
        state = None
        if self.memory is not None:
            # load_for_prompt is defined in the memory cell above
            kw = {"ttl_days": self.ttl_days}
            if self.today is not None:
                kw["today"] = self.today
            state, mem_block = load_for_prompt(self.memory, self.user_id, **kw)
            if mem_block:
                system = SYSTEM + "\n\n" + mem_block
                result.memory_used = True
        messages = [{"role": "system", "content": system},
                    {"role": "user", "content": question}]
        for i in range(self.max_iter):
            result.iterations = i + 1
            result.prompt_chars += sum(len(str(m.get("content") or "")) for m in messages)
            resp = self.llm.complete(messages, self.tools)
            if resp.kind == "final":
                result.answer = resp.content or ""
                result.stop_reason = "final"
                break
            # tool call
            name, args = resp.tool_name, resp.tool_args or {}
            obs, status = self._execute(name, args)
            result.steps.append(Step(tool=name, args=args, observation=obs, status=status))
            call_id = f"call_{i}"
            messages.append({"role": "assistant", "content": None,
                             "tool_calls": [{"id": call_id, "type": "function",
                                             "function": {"name": name,
                                                          "arguments": json.dumps(args)}}]})
            messages.append({"role": "tool", "tool_call_id": call_id, "content": obs})
        else:
            # max-iteration guard fired
            result.stop_reason = "max_iter"
            result.answer = ("(stopped after the maximum number of steps — I couldn't reach a "
                             "final answer safely.)")
        # ── user-controlled memory: honour 'remember …' / 'forget …' ──
        if self.memory is not None and state is not None:
            # update_from_turn is defined in the memory cell above
            kw = {"today": self.today} if self.today is not None else {}
            result.memory_note = update_from_turn(self.memory, state, question, **kw)
        return result


# ─── assembly ───────────────────────────────────────────────────────────────
def default_tools(offline: bool = True) -> list[Tool]:
    # the four Tool classes are defined in the cells above
    return [SearchWebTool(offline=offline), QuerySqlTool(), CalculateTool(), SendEmailTool()]


def build_agent(offline: bool = True, use_llm: bool = False, hardened: bool = True,
                max_iter: int = 5, memory: bool = False, user_id: str = "default",
                ttl_days: int = 30, today: Any = None) -> ReActAgent:
    """Default: offline MockLLM + hardened with the DefensePolicy. Set memory=True to
    attach the long-term state layer (SQLite-backed)."""
    tools = default_tools(offline=offline)
    if use_llm:
        # OpenAILLM is defined above
        llm: Any = OpenAILLM()
    else:
        llm = MockLLM()
    policy = DefensePolicy() if hardened else None
    store = None
    if memory:
        # MemoryStore is defined in the memory cell above
        store = MemoryStore()
    return ReActAgent(llm, tools, max_iter=max_iter, defenses=policy,
                      memory=store, user_id=user_id, ttl_days=ttl_days, today=today)


**Pre-seed a user's memory, then run the agent for that user** — it loads and injects the memory automatically:

In [15]:
store = MemoryStore(); store.clear('asha')
s = UserState(user_id='asha'); s.set('preferred_format', 'bullets', today=JAN15); store.save_state(s)
state, block = load_for_prompt(store, 'asha', today=JAN15)
print('injected into the system prompt:\n' + block + '\n')

agent = build_agent(offline=True, memory=True, user_id='asha', hardened=False, today=JAN15)
r = agent.run('what is the leave policy?')
print('memory_used:', r.memory_used, '| prompt_chars (cost proxy):', r.prompt_chars)
r2 = agent.run('remember that preferred_format is JSON')
print('remember command →', r2.memory_note)

injected into the system prompt:
What you remember about this user:
- preferred format: bullets

memory_used: True | prompt_chars (cost proxy): 245
remember command → remembered preferred_format = JSON


## Recap

Five kinds of memory (you need one or two); three patterns (state is the default); **default to no memory**; and a small state-based layer — `UserState` + SQLite + TTL + user-controlled forget — that the agent injects into its prompt. Next, **Notebook 2** measures whether that memory actually *helps or hurts*.